***ДЗ. Построение списка похожих слов***

Написать функции, возвращающие список слов, близких к целевому

а) из векторного пространства модели
б)  из списка слов, содержащихся в некотором файле

Применить эти функции и сравнить полученные результаты и методы.
Для удобного сравнения результатов можно в функциях задавать еще параметры (например, косинусное расстояние, количество слов в списке).
Можно ли назвать полученный список списком синонимов?

Ответ к заданию: ноутбук с программой (написанные функции должны быть выделены, используемые методы описаны) и текстовой ответ на вопросы (не в ноутбуке)

Пояснение к заданию

а) Список похожих слов модели выдает метод most_similar, но, выполняя аудиторную работу, Вы должны были увидеть, что самыми близкими могут оказаться словоформы целевого слова, а также целевое слово, написанное с ошибками и опечатками.
Решение – брать большое N, исключить из полученного списка словоформы (можно не пользоваться средствами NLTK, а считать словоформами слова с одинаковым началом), а слова со знаками препинания внутри исключить, например, с помощью регулярных выражений.

Облегченный вариант задания: производить отсев ненужных слов вручную

 б) В приложенном файле есть код, пользуясь  которым, можно, вручную изменяя уровень близости, подобрать список слов заданного размера. Используя его, напишите фрагмент программы, с помощью которого можно получить список заданной длины автоматически или список с другими параметрами, который удобно автоматически сравнивать со списком из задания а)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import gensim
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('.')

In [ ]:
try:
    from utils import download_model
    print("Функция download_model успешно импортирована")
except ImportError:
    print("Файл utils.py не найден.")

In [ ]:
DATA_DIR = 'data'
RANDOM_STATE = 42

download_model(data_dir=DATA_DIR)

In [ ]:
try:
    wv_embeddings = gensim.models.fasttext.FastTextKeyedVectors.load(
        DATA_DIR + "/model/model.model"
    )
    print(f"Размер словаря: {len(wv_embeddings.key_to_index)} слов")
    print(f"Размерность эмбеддингов: {wv_embeddings.vector_size}")
except Exception as e:
    print(f"Ошибка при загрузке модели: {e}")

In [ ]:
from gensim.models import FastText
    sentences = [["кошка", "сидит", "на", "ковре"], 
                 ["собака", "бежит", "по", "полю"],
                 ["кошка", "и", "собака", "друзья"]]
    model = FastText(sentences, vector_size=10, window=3, min_count=1, workers=4)
    wv_embeddings = model.wv

In [ ]:
def normalize_embedding(embedding):
    """
    Нормализация вектора (приведение к единичной длине)
    
    Args:
        embedding: Вектор эмбеддинга
        
    Returns:
        Нормализованный вектор
    """
    emb = np.array(embedding)
    norm = np.linalg.norm(emb)
    if norm == 0:
        return emb
    return emb / norm

def get_word_embedding(word, normalize=True):
    """
    Получение эмбеддинга слова с возможностью нормализации
    
    Args:
        word: Слово
        normalize: Нужно ли нормализовать вектор
        
    Returns:
        Вектор эмбеддинга или None, если слово не найдено
    """
    try:
        embedding = wv_embeddings[word]
        if normalize:
            embedding = normalize_embedding(embedding)
        return embedding
    except KeyError:
        return None